In [5]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
processed_dir = project_root / 'data' / 'processed'

candidates = pd.read_csv(processed_dir / 'candidates_raw_copy.csv')
jobs = pd.read_csv(processed_dir / 'jobs_raw_copy.csv')

candidate_markers = {'Name', 'Skills', 'Years_Experience'}
job_markers = {'Job Title', 'Required Skills', 'Experience Level'}

# If files were saved/swapped incorrectly, correct assignment by schema.
if candidate_markers.issubset(set(jobs.columns)) and job_markers.issubset(set(candidates.columns)):
    candidates, jobs = jobs.copy(), candidates.copy()

# ══════════════════════════════════════════════════════════════════
# CLEAN CANDIDATES
# ══════════════════════════════════════════════════════════════════

def clean_name(name):
    """Remove titles like 'MD', 'PhD' etc."""
    if pd.isna(name):
        return name
    return re.sub(r'\s+(MD|PhD|Jr|Sr|II|III)$', '', str(name).strip())

def clean_skills(skills_str):
    """Normalize skills: strip whitespace, fix typos, lowercase for matching."""
    if pd.isna(skills_str) or str(skills_str).strip() == '':
        return ''
    skills = [s.strip().lower() for s in str(skills_str).split(',')]
    # Remove empty strings
    skills = [s for s in skills if s]
    return ', '.join(skills)

def map_experience_to_level(years):
    """Convert numeric years to seniority label."""
    if pd.isna(years):
        return 'Unknown'
    years = int(years)
    if years == 0:
        return 'Entry Level'
    elif years <= 3:
        return 'Entry Level'
    elif years <= 6:
        return 'Mid Level'
    else:
        return 'Senior Level'

In [6]:
# Apply cleaning
candidates['Name'] = candidates['Name'].apply(clean_name)
candidates['Skills_Clean'] = candidates['Skills'].apply(clean_skills)
candidates['Experience_Level'] = candidates['Years_Experience'].apply(map_experience_to_level)

In [9]:
print('Candidates columns:', candidates.columns.tolist())
print('Jobs columns:', jobs.columns.tolist())

Candidates columns: ['Name', 'Email', 'Phone', 'University', 'Graduation_Year', 'Years_Experience', 'Job_Role', 'Skills', 'Resume_Text', 'Skills_Clean', 'Experience_Level']
Jobs columns: ['Job Title', 'Company', 'Location', 'Experience Level', 'Salary', 'Industry', 'Required Skills']


In [10]:
# Fill missing emails/phones
candidates['Email'] = candidates['Email'].fillna('Not provided')
candidates['Phone'] = candidates['Phone'].fillna('Not provided')

# Drop rows with no name
candidates = candidates.dropna(subset=['Name'])

print(f"Candidates after cleaning: {len(candidates)}")
print(candidates[['Name', 'Skills_Clean', 'Experience_Level']].head(8))


Candidates after cleaning: 815
              Name                                       Skills_Clean  \
0    Tara Gonzalez  python, machine learning, numpy, scikit-learn,...   
1      Jared Quinn  numpy, scikit-learn, statistics, tensorflow, m...   
2       Jane Woods   tensorflow, python, numpy, machine learning, nlp   
3   Julie Chambers  scikit-learn, statistics, data visualization, ...   
4   Tonya Campbell             python, numpy, sql, statistics, pandas   
5  Aaron Rodriguez  machine learning, data visualization, sql, pyt...   
6   Louis Williams  tensorflow, python, pandas, scikit-learn, stat...   
7    Jason Calhoun    nlp, data visualization, numpy, sql, tensorflow   

  Experience_Level  
0      Entry Level  
1        Mid Level  
2        Mid Level  
3      Entry Level  
4      Entry Level  
5      Entry Level  
6        Mid Level  
7        Mid Level  


In [11]:
# ══════════════════════════════════════════════════════════════════
# CLEAN JOBS
# ══════════════════════════════════════════════════════════════════

def clean_required_skills(skills_str):
    """Same normalization for job required skills."""
    if pd.isna(skills_str) or str(skills_str).strip() == '':
        return ''
    skills = [s.strip().lower() for s in str(skills_str).split(',')]
    skills = [s for s in skills if s]
    return ', '.join(skills)

jobs['Required_Skills_Clean'] = jobs['Required Skills'].apply(clean_required_skills)
jobs['Job_Title_Clean'] = jobs['Job Title'].str.strip().str.lower()

# Standardize experience level spelling
jobs['Experience Level'] = jobs['Experience Level'].str.strip()

print(f"\nJobs after cleaning: {len(jobs)}")
print(jobs[['Job Title', 'Required_Skills_Clean', 'Experience Level']].head(8))

# ══════════════════════════════════════════════════════════════════
# SAVE
# ══════════════════════════════════════════════════════════════════

candidates.to_csv('../data/processed/candidates_clean.csv', index=False)
jobs.to_csv('../data/processed/jobs_clean.csv', index=False)
print("\nCleaned files saved.")


Jobs after cleaning: 50000
                         Job Title  \
0              Early years teacher   
1         Counselling psychologist   
2        Radio broadcast assistant   
3     Designer, exhibition/display   
4  Psychotherapist, dance movement   
5              Early years teacher   
6               Academic librarian   
7                Quantity surveyor   

                               Required_Skills_Clean Experience Level  
0                                    pharmaceuticals     Senior Level  
1                   google ads, seo, content writing        Mid Level  
2  patient care, nursing, medical research, pharm...        Mid Level  
3                                   machine learning     Senior Level  
4         nursing, medical research, pharmaceuticals      Entry Level  
5                  financial modeling, risk analysis        Mid Level  
6                                    pharmaceuticals      Entry Level  
7                                       java, python 